# Sequential Deep Learning: Bidirectional LSTM with TensorBoard

**Task:** Predict building electrical power consumption 24 hours ahead (144 steps at 10-minute resolution) using a PyTorch Bidirectional Long Short-Term Memory (BiLSTM) network.  
**Data:** `library_featured_v2.csv` (engineered in notebook 05) with 27 temporal, calendar, occupancy, and autoregressive features.  
**Benchmark Target:** Compare sequential memory against tabular MLP and gradient boosted tree baselines (LightGBM, Extra Trees, XGBoost) using the exact same chronological 70/30 test split.

---

### Sequence Modeling Formulation in University BMS
Unlike standard feedforward networks that process each time point in isolation, recurrent architectures model continuous temporal trajectories:
1. **Recurrent Gating Dynamics:**
   An LSTM unit computes input ($i_t$), forget ($f_t$), and output ($o_t$) gates to regulate long-term memory cell retention:
   $$\begin{aligned}
   f_t &= \sigma(W_f x_t + U_f h_{t-1} + b_f) \\
   i_t &= \sigma(W_i x_t + U_i h_{t-1} + b_i) \\
   \tilde{c}_t &= \tanh(W_c x_t + U_c h_{t-1} + b_c) \\
   c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \\
   o_t &= \sigma(W_o x_t + U_o h_{t-1} + b_o) \\
   h_t &= o_t \odot \tanh(c_t)
   \end{aligned}$$
2. **Bidirectional Temporal Encoding:**
   We process a sliding sequence window of historical context ($L = 36$ steps, representing the preceding 6 hours of continuous 10-minute dynamics). In conjunction with our 24-hour and 7-day autoregressive lag features (`power_lag_144`, `power_lag_1008`), the forward LSTM tracks chronological build-up while the backward LSTM tracks contextual proximity.
3. **Regression Output Projection:**
   The concatenated forward and backward hidden states $\mathbf{h}_{\text{last}} \in \mathbb{R}^{128}$ are projected through a dense regression head: `Linear(64)` $\to$ `ReLU` $\to$ `Dropout(0.2)` $\to$ `Linear(1)`. The output has **no activation function**, producing unconstrained continuous power forecasts in Watts ($W$).

## 1. Imports & Setup

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ensure utils path is discoverable
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))
if "." not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

from utils.tb_logger import TorchTBLogger

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Computation Device: {DEVICE}")

PyTorch Version: 2.14.0+cpu
Computation Device: cpu


## 2. Load Engineered Feature Matrix

In [2]:
DATA_PATH = "library_featured_v2.csv" if os.path.exists("library_featured_v2.csv") else "../library_featured_v2.csv"
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {DATA_PATH}: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded library_featured_v2.csv: 41,248 rows x 37 columns


## 3. Feature Selection & Chronological Train/Val/Test Split

In [3]:
FEATURE_COLUMNS = [
    # Baseline power & occupancy
    "power", "occupancy_count", "occupancy_roll_mean_6", "occupancy_diff_1", "rooms_occupied_est",
    # Cyclical temporal features
    "hour_sin", "hour_cos", "dayofweek_sin", "dayofweek_cos", "month_sin", "month_cos", "dayofyear_sin", "dayofyear_cos",
    # BMS operational schedule flags
    "weekend", "is_class_hour", "is_evening_study", "is_night", "is_exam_month", "is_vacation_month",
    # Autoregressive lags & rolling stats
    "power_lag_144", "power_lag_288", "power_lag_1008", "occupancy_lag_144",
    "power_roll_mean_144", "power_roll_std_144", "power_roll_min_144", "power_roll_max_144"
]
TARGET_COLUMN = "target"

# Primary 70 / 30 chronological split
test_split_idx = int(len(df) * 0.70)
train_val_df = df.iloc[:test_split_idx].reset_index(drop=True)
test_df = df.iloc[test_split_idx:].reset_index(drop=True)

# Train / Validation sub-split (85% train / 15% val)
val_split_idx = int(len(train_val_df) * 0.85)
train_df = train_val_df.iloc[:val_split_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_split_idx:].reset_index(drop=True)

print(f"Features: {len(FEATURE_COLUMNS)} variables")
print(f"Train sub-split: {len(train_df):>6,} rows | {train_df['timestamp'].min()} to {train_df['timestamp'].max()}")
print(f"Validation split:{len(val_df):>6,} rows | {val_df['timestamp'].min()} to {val_df['timestamp'].max()}")
print(f"Test split:      {len(test_df):>6,} rows | {test_df['timestamp'].min()} to {test_df['timestamp'].max()}")

Features: 27 variables
Train sub-split: 24,542 rows | 2014-02-26 10:30:00+05:30 to 2016-08-10 12:00:00+05:30
Validation split: 4,331 rows | 2016-08-10 12:10:00+05:30 to 2016-11-20 12:50:00+05:30
Test split:      12,375 rows | 2016-11-20 13:00:00+05:30 to 2017-11-02 23:50:00+05:30


## 4. Normalization and Sliding Window Sequence Dataset

For sequence models:
- Normalization parameters are computed **strictly on the training sub-split**.
- We construct a sliding window `Dataset` where each sample consists of:
  - Input sequence $X_{i:i+L} \in \mathbb{R}^{L \times D}$ (window of length $L = 36$, representing the preceding 6 hours).
  - Target $y_{i+L-1} \in \mathbb{R}$ (the 24-hour ahead power reading corresponding to the final timestep).

In [4]:
scaler_x = StandardScaler()
scaler_y = StandardScaler()

# Fit scalers strictly on training data
X_train_scaled = scaler_x.fit_transform(train_df[FEATURE_COLUMNS].values)
y_train_scaled = scaler_y.fit_transform(train_df[TARGET_COLUMN].values.reshape(-1, 1)).ravel()

X_val_scaled = scaler_x.transform(val_df[FEATURE_COLUMNS].values)
y_val_scaled = scaler_y.transform(val_df[TARGET_COLUMN].values.reshape(-1, 1)).ravel()

X_test_scaled = scaler_x.transform(test_df[FEATURE_COLUMNS].values)
y_test_scaled = scaler_y.transform(test_df[TARGET_COLUMN].values.reshape(-1, 1)).ravel()

SEQ_LEN = 36  # 36 steps of 10 min = 6 hours of continuous lookback

class TimeSeriesSequenceDataset(Dataset):
    def __init__(self, X, y, seq_len=36):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.seq_len = seq_len
        
    def __len__(self):
        return len(self.X) - self.seq_len + 1
        
    def __getitem__(self, idx):
        x_window = self.X[idx : idx + self.seq_len]
        y_target = self.y[idx + self.seq_len - 1]
        return x_window, y_target

BATCH_SIZE = 256

train_dataset = TimeSeriesSequenceDataset(X_train_scaled, y_train_scaled, seq_len=SEQ_LEN)
val_dataset   = TimeSeriesSequenceDataset(X_val_scaled,   y_val_scaled,   seq_len=SEQ_LEN)
test_dataset  = TimeSeriesSequenceDataset(X_test_scaled,  y_test_scaled,  seq_len=SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Sequence Datasets ready (seq_len={SEQ_LEN}):")
print(f"  Train: {len(train_dataset):,} sequences ({len(train_loader)} batches)")
print(f"  Val:   {len(val_dataset):,} sequences ({len(val_loader)} batches)")
print(f"  Test:  {len(test_dataset):,} sequences ({len(test_loader)} batches)")

Sequence Datasets ready (seq_len=36):
  Train: 24,507 sequences (96 batches)
  Val:   4,296 sequences (17 batches)
  Test:  12,340 sequences (49 batches)


## 5. PyTorch Bidirectional LSTM Architecture

The model architecture:
- Input shape: $(B, L, D) = (B, 36, 27)$
- `nn.LSTM`: 1 layer, `hidden_size=64`, `bidirectional=True`.
- Temporal State Extraction: Concatenates forward and backward hidden states $\to \mathbb{R}^{128}$.
- Regression Head: `Linear(128, 64)` $\to$ `ReLU()` $\to$ `Dropout(0.2)` $\to$ `Linear(64, 1)`.
- Output: Single continuous value with **no activation** (linear regression).

In [5]:
class BMS_BiLSTM(nn.Module):
    def __init__(self, in_features, hidden_dim=64, bidirectional=True, dropout=0.2):
        super().__init__()
        self.bidirectional = bidirectional
        self.hidden_dim = hidden_dim
        
        self.lstm = nn.LSTM(
            input_size=in_features,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=bidirectional
        )
        
        fc_in = hidden_dim * 2 if bidirectional else hidden_dim
        self.head = nn.Sequential(
            nn.Linear(fc_in, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)  # Linear output (no activation)
        )
        
    def forward(self, x):
        # x: [B, L, D]
        _, (hn, _) = self.lstm(x)
        # hn: [num_layers * num_directions, B, hidden_dim]
        if self.bidirectional:
            h_fwd = hn[-2, :, :]
            h_bwd = hn[-1, :, :]
            h_repr = torch.cat([h_fwd, h_bwd], dim=1)
        else:
            h_repr = hn[-1, :, :]
            
        out = self.head(h_repr)
        return out.squeeze(-1)

model = BMS_BiLSTM(in_features=len(FEATURE_COLUMNS), hidden_dim=64, bidirectional=True, dropout=0.2).to(DEVICE)
print(model)
print(f"Total Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

BMS_BiLSTM(
  (lstm): LSTM(27, 64, batch_first=True, bidirectional=True)
  (head): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)
Total Trainable Parameters: 55,937


## 6. Initialize TensorBoard Logger & Computational Graph

In [6]:
log_root = "../runs" if os.path.basename(os.getcwd()) == "notebooks" else "runs"
os.makedirs(log_root, exist_ok=True)

sample_x, _ = next(iter(train_loader))
sample_x = sample_x.to(DEVICE)

tb = TorchTBLogger(
    model_name="LSTM",
    model=model,
    sample_input=sample_x,
    log_root=log_root
)

[TB] 'LSTM' -> ../runs\LSTM
[TB] Launch: tensorboard --logdir=../runs/


[TB] Model graph logged.


## 7. Model Training & Per-Epoch TensorBoard Logging

In [7]:
EPOCHS = 15
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
criterion = nn.MSELoss()

best_val_loss = float("inf")
checkpoint_path = os.path.join(log_root, "best_LSTM.pt")

print(f"Training BiLSTM on {DEVICE} for {EPOCHS} epochs...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        preds = model(x_b)
        loss = criterion(preds, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())
        
    avg_train_loss = np.mean(train_losses)
    
    model.eval()
    val_losses = []
    val_preds_list = []
    val_targets_list = []
    with torch.no_grad():
        for x_b, y_b in val_loader:
            x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
            preds = model(x_b)
            loss = criterion(preds, y_b)
            val_losses.append(loss.item())
            val_preds_list.extend(preds.cpu().numpy())
            val_targets_list.extend(y_b.cpu().numpy())
            
    avg_val_loss = np.mean(val_losses)
    current_lr = scheduler.get_last_lr()[0]
    
    val_preds_W = scaler_y.inverse_transform(np.array(val_preds_list).reshape(-1, 1)).ravel()
    val_targets_W = scaler_y.inverse_transform(np.array(val_targets_list).reshape(-1, 1)).ravel()
    
    val_mae = mean_absolute_error(val_targets_W, val_preds_W)
    val_rmse = np.sqrt(mean_squared_error(val_targets_W, val_preds_W))
    val_r2 = r2_score(val_targets_W, val_preds_W)
    
    tb.log_epoch(
        epoch=epoch,
        train_loss=avg_train_loss,
        val_loss=avg_val_loss,
        y_val=val_targets_W,
        y_pred=val_preds_W,
        lr=current_lr
    )
    
    if epoch % 5 == 0:
        tb.log_gradients(model, epoch)
        
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), checkpoint_path)
        
    scheduler.step()
    
    print(f"Epoch {epoch:>2d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val MAE: {val_mae:,.1f} W | Val R²: {val_r2:.4f} | LR: {current_lr:.2e}")

print(f"\nTraining complete. Best checkpoint saved to {checkpoint_path}")

Training BiLSTM on cpu for 15 epochs...


Epoch  1/15 | Train Loss: 0.3964 | Val Loss: 0.4244 | Val MAE: 3,245.4 W | Val R²: 0.6315 | LR: 1.00e-03


Epoch  2/15 | Train Loss: 0.1895 | Val Loss: 0.4504 | Val MAE: 3,321.2 W | Val R²: 0.6091 | LR: 9.89e-04


Epoch  3/15 | Train Loss: 0.1441 | Val Loss: 0.4299 | Val MAE: 3,217.4 W | Val R²: 0.6263 | LR: 9.57e-04


Epoch  4/15 | Train Loss: 0.1123 | Val Loss: 0.4680 | Val MAE: 3,399.6 W | Val R²: 0.5927 | LR: 9.05e-04


...

Epoch 11/15 | Train Loss: 0.0563 | Val Loss: 0.4929 | Val MAE: 3,489.9 W | Val R²: 0.5707 | LR: 2.58e-04


Epoch 12/15 | Train Loss: 0.0536 | Val Loss: 0.4976 | Val MAE: 3,515.4 W | Val R²: 0.5665 | LR: 1.74e-04


Epoch 13/15 | Train Loss: 0.0510 | Val Loss: 0.4956 | Val MAE: 3,503.5 W | Val R²: 0.5682 | LR: 1.05e-04


Epoch 14/15 | Train Loss: 0.0505 | Val Loss: 0.4993 | Val MAE: 3,510.3 W | Val R²: 0.5649 | LR: 5.28e-05


Epoch 15/15 | Train Loss: 0.0498 | Val Loss: 0.5005 | Val MAE: 3,517.9 W | Val R²: 0.5638 | LR: 2.08e-05

Training complete. Best checkpoint saved to ../runs\best_LSTM.pt


## 8. Final Test Evaluation & Benchmark Comparison

In [8]:
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model.eval()

# Predict on test sequences
test_preds_list = []
with torch.no_grad():
    for x_b, _ in test_loader:
        x_b = x_b.to(DEVICE)
        preds = model(x_b)
        test_preds_list.extend(preds.cpu().numpy())

test_preds_W = scaler_y.inverse_transform(np.array(test_preds_list).reshape(-1, 1)).ravel()
test_targets_W = test_df[TARGET_COLUMN].values[SEQ_LEN - 1:]

# Predict on training sequences for gap check
train_val_dataset = TimeSeriesSequenceDataset(
    scaler_x.transform(train_val_df[FEATURE_COLUMNS].values),
    scaler_y.transform(train_val_df[TARGET_COLUMN].values.reshape(-1, 1)).ravel(),
    seq_len=SEQ_LEN
)
train_val_loader = DataLoader(train_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
train_preds_list = []
with torch.no_grad():
    for x_b, _ in train_val_loader:
        x_b = x_b.to(DEVICE)
        preds = model(x_b)
        train_preds_list.extend(preds.cpu().numpy())

train_preds_W = scaler_y.inverse_transform(np.array(train_preds_list).reshape(-1, 1)).ravel()
train_targets_W = train_val_df[TARGET_COLUMN].values[SEQ_LEN - 1:]

test_mae = mean_absolute_error(test_targets_W, test_preds_W)
test_rmse = np.sqrt(mean_squared_error(test_targets_W, test_preds_W))
test_r2 = r2_score(test_targets_W, test_preds_W)
train_r2 = r2_score(train_targets_W, train_preds_W)
r2_gap = train_r2 - test_r2

print("=" * 55)
print(f"  PyTorch BiLSTM Final Benchmark Results (Test Set)")
print("=" * 55)
print(f"  Test MAE:   {test_mae:,.1f} W")
print(f"  Test RMSE:  {test_rmse:,.1f} W")
print(f"  Test R²:    {test_r2:.4f}")
print(f"  Train R²:   {train_r2:.4f}")
print(f"  R² Gap:     {r2_gap:.4f}")
print("=" * 55)

  PyTorch BiLSTM Final Benchmark Results (Test Set)
  Test MAE:   3,849.7 W
  Test RMSE:  5,183.3 W
  Test R²:    0.5758
  Train R²:   0.7607
  R² Gap:     0.1849


## 9. Log Final Metrics, Prediction Plots & HParams to TensorBoard

In [9]:
tb.log_final(
    y_train=train_targets_W,
    y_pred_train=train_preds_W,
    y_test=test_targets_W,
    y_pred_test=test_preds_W,
    hparams={
        "architecture": "BiLSTM (1-layer, 64-dim)",
        "seq_len": SEQ_LEN,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr_init": 1e-3,
        "dropout": 0.2,
        "optimizer": "AdamW",
        "output_act": "Linear"
    }
)

tb.log_prediction_plots(y_test=test_targets_W, y_pred=test_preds_W, n_plot=500)
tb.log_scatter_plot(y_test=test_targets_W, y_pred=test_preds_W)

tb.close()
print("TensorBoard logging complete. Logs written to:", log_root)


[LSTM] Train  MAE=2352W  R2=0.761
[LSTM] Test   MAE=3850W  R2=0.576


[TB] Writer closed: 'LSTM'
TensorBoard logging complete. Logs written to: ../runs


## 10. Committee Presentation Summary: Model Comparison

| Model | Model Family | Test MAE (W) | Test RMSE (W) | Test R² | Train-Test R² Gap |
|---|---|---|---|---|---|
| **LightGBM** | Gradient Boosted Trees | ~3,340 | ~4,580 | ~0.672 | 0.210 |
| **Extra Trees** | Randomized Ensembles | ~3,410 | ~4,690 | ~0.655 | 0.280 |
| **XGBoost** | Gradient Boosted Trees | 3,180 | 4,405 | 0.693 | 0.170 |
| **PyTorch MLP** | Deep Tabular NN | ~3,469 | ~4,636 | ~0.660 | 0.168 |
| **PyTorch BiLSTM** | Sequential Recurrent NN | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* |

Launch TensorBoard to inspect side-by-side:
```bash
tensorboard --logdir=runs/
```